In [1]:
import pandas
import numpy as np
import pandas as pd
from huggingface_hub.keras_mixin import keras
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv2D,MaxPooling2D,Flatten,Dense,Input,LSTM,Embedding
from tensorflow.keras.utils import to_categorical
import keras
import os
import unicodedata
import re

In [2]:
from datasets import load_dataset

ds = load_dataset("opus100", "en-fa")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/opus100/resolve/main/README.md (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001904FAEB620>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: e2100ecb-c2f1-40a9-9185-af7cf898fd7f)')' thrown while requesting HEAD https://huggingface.co/datasets/opus100/resolve/main/README.md
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/opus100/resolve/main/README.md (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001904FB66AD0>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 69066751-61bf-4a0a-8a9c-4176e6315c91)')' thrown while requesting HEAD https://huggingface.co/datasets/opus100/resolve/main/README.md
Retrying in 2s [Retry 2/5].
'(MaxR

In [3]:
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD',s)if unicodedata.category(c) !='MN')

In [4]:
en_list=[]
fa_list=[]
train_data=ds['train'].select(range(3000))
for   items in  train_data:
    en = items['translation']['en'].lower().strip()
    en= re.sub(r"([.!?~,])", r" \1", en)
    en= re.sub(r'([" "])+', " ", en)
    en= '<start> ' + en + ' <end>'

    fa=items['translation']['fa'].lower().strip()
    fa=re.sub(r"\s+",' ',fa)
    en_list.append(en)
    fa_list.append(fa)

In [5]:
df=pd.DataFrame({'fa':fa_list,'en':en_list})
df. to_csv("fa-en-train.csv",index=False,encoding='utf-8')

In [6]:
from transformers import AutoTokenizer
import tensorflow as tf
batch_size=10
x_batch=[]
decoder_inputs_batch=[]
decoder_targets_batch=[]
max_length=50
tokenizer_en=AutoTokenizer.from_pretrained("t5-small",local_files_only=True)
tokenizer_fa=AutoTokenizer.from_pretrained(r"C:/local_model/t5-small",local_files_only=True)

for i in range(0,len(df),batch_size):
    en_inputs=tokenizer_en(df['en'].iloc[i:i+batch_size].tolist(),
                       padding="max_length",
                        max_length=max_length,
                       truncation=True,
                       return_tensors="tf",)

    fa_labels=tokenizer_fa(df['fa'].iloc[i:i+batch_size].tolist(),
                       padding="max_length",
                        max_length=max_length,
                       truncation=True,
                       return_tensors="tf")

    x_batch.append(en_inputs['input_ids'])

    decoder_inputs=fa_labels['input_ids'][:,:-1]
    decoder_targets=fa_labels['input_ids'][:,1:]

    decoder_inputs_batch.append(decoder_inputs)
    decoder_targets_batch.append(decoder_targets)

x_train=tf.concat(x_batch,axis=0)
decoder_inputs_data=tf.concat(decoder_inputs_batch,axis=0)
decoder_targets_data=tf.concat(decoder_targets_batch,axis=0)

print(f"x_train{x_train.shape}")
print(f"decoder inputs:{decoder_inputs_data.shape}")
print(f"decoder targets:{decoder_targets_data.shape}")
print(x_batch[0].shape,decoder_inputs_batch[0].shape,decoder_targets_batch[0].shape)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


x_train(3000, 50)
decoder inputs:(3000, 49)
decoder targets:(3000, 49)
(10, 50) (10, 49) (10, 49)


In [7]:
embdding_dim=100
units=512
vocab_size_en=tokenizer_en.vocab_size
vocab_size_fa=tokenizer_fa.vocab_size


encoder_inputs=Input(shape=(None,))
enc_emb=Embedding(vocab_size_en,embdding_dim,)(encoder_inputs)
encoder_output,state_h,state_c=LSTM(units,return_state=True)(enc_emb)
encoder_states=[state_h,state_c]


decoder_inputs=Input(shape=(None,))
dec_emb=Embedding(vocab_size_fa,embdding_dim,)(decoder_inputs)
decoder_lstm=LSTM(units,return_state=True,return_sequences=True)
decoder_outputs,_,_=decoder_lstm(dec_emb,initial_state=encoder_states)
decoder_dense=Dense(vocab_size_fa,activation="softmax")
decoder_outputs=decoder_dense(decoder_outputs)


In [8]:
model=Model([encoder_inputs,decoder_inputs],decoder_outputs)

In [9]:
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 100) │  3,210,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 100) │  3,210,000 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,255,424 │ embedding[0][0]   │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │  1,255,424 │ embedding_1[0][0… │
│                     │ 512), (None,      │            │ lstm[0][1],       │
│                     │ 512), (None,      │            │ lstm[0][2]        │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 16,467,300 │ lstm_1[0][0]      │
│                     │ 32100)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,398,148 (96.89 MB)

 Trainable params: 25,398,148 (96.89 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history=model.fit([x_train,decoder_inputs_data],
          np.expand_dims(decoder_targets_data,axis=-1),
        batch_size=10,epochs=10,
          validation_split=0.1,
          verbose=1)

Epoch 1/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 77s 279ms/step - accuracy: 0.8444 - loss: 0.7843 - val_accuracy: 0.9401 - val_loss: 0.3852
Epoch 2/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 78s 287ms/step - accuracy: 0.9601 - loss: 0.2025 - val_accuracy: 0.9569 - val_loss: 0.2176
Epoch 3/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 79s 293ms/step - accuracy: 0.9663 - loss: 0.1567 - val_accuracy: 0.9607 - val_loss: 0.2096
Epoch 4/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 77s 286ms/step - accuracy: 0.9670 - loss: 0.1503 - val_accuracy: 0.9612 - val_loss: 0.2033
Epoch 5/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 78s 289ms/step - accuracy: 0.9672 - loss: 0.1484 - val_accuracy: 0.9609 - val_loss: 0.2110
Epoch 6/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 79s 291ms/step - accuracy: 0.9674 - loss: 0.1461 - val_accuracy: 0.9609 - val_loss: 0.2016
Epoch 7/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 79s 294ms/step - accuracy: 0.9674 - loss: 0.1438 - val_accuracy: 0.9618 - val_loss: 0.1957
Epoch 8/10
270/270 ━━━━━━━━━━━━━━━━━━━━ 78s 289ms/step - accuracy: 0.9665 - loss: 0

In [12]:
model.save("Translator.keras")